# Branch Performance View
create a branch performance view with columns
- branch_code
- branch_name
- total_customers
- total_accounts
- total_deposits
- total_transactions
- total_transaction_amount

In [0]:
CREATE OR REPLACE VIEW neo_bank.gold.vw_branch_performance AS

WITH customer_agg AS (
    select count(customer_id) as total_customers,home_branch_sk from neo_bank.gold.dim_customers group by home_branch_sk
),

accounts_agg AS (
    select count(account_id) as total_accounts,sum(balance) as total_deposits,account_branch_sk from neo_bank.gold.dim_accounts group by account_branch_sk
),

transactions_agg AS (
    select count(neo_bank.gold.fact_transactions.txn_id) as total_transactions,
    sum(neo_bank.gold.fact_transactions.amount) as total_transaction_amount,
    neo_bank.gold.dim_accounts.account_branch_sk 
    from neo_bank.gold.fact_transactions 
    INNER JOIN neo_bank.gold.dim_accounts 
    on neo_bank.gold.fact_transactions.account_sk==neo_bank.gold.dim_accounts.account_sk
    group by neo_bank.gold.dim_accounts.account_branch_sk
)

select
    b.branch_code,
    b.branch_name,
    c.total_customers,
    a.total_accounts,
    a.total_deposits,
    t.total_transactions,
    t.total_transaction_amount
from customer_agg c 
left join accounts_agg a on c.home_branch_sk = a.account_branch_sk
left join transactions_agg t on a.account_branch_sk = t.account_branch_sk
left join neo_bank.gold.dim_branches b on t.account_branch_sk = b.branch_sk
order by b.branch_code


In [0]:
select * from neo_bank.gold.vw_branch_performance